<a href="https://colab.research.google.com/github/acerNZ/HAL/blob/master/ConvertBDD_Traditional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --------------------------------------------------------------
#  GHERKIN → MoSCoW (full hierarchical document)  –  Colab ready
# --------------------------------------------------------------
# 1. Install nothing – only built-in + pandas + ipywidgets
# 2. Upload Excel/CSV with column “Acceptance Criteria”
# 3. Download the same file with an extra column “MoSCoW Requirements”
# --------------------------------------------------------------

import re, io, base64
import pandas as pd
from collections import OrderedDict
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown

# -------------------  CONFIG  -------------------
MUST_KEYWORDS = ["must","require","mandatory","lock","save","display","allow",
                 "navigate cloud","prompt","ask","warning","validate","cannot","not allow"]
SHOULD_KEYWORDS = ["should","warning","recommend"]
COULD_ENHANCEMENTS = ["audit","history","bulk","filter","advanced","quickview"]

# -------------------  PARSER  -------------------
def parse_gherkin(text):
    """Return list of dicts – one per Given/When/Then block."""
    blocks = []
    current = []
    for line in text.splitlines():
        line = line.strip()
        if not line:                     # blank line → end of block
            if current:
                blocks.append("\n".join(current))
                current = []
            continue
        if line.lower().startswith(("given","when","then","and")):
            current.append(line)
        else:                            # continuation of previous line
            if current:
                current[-1] += " " + line
    if current:
        blocks.append("\n".join(current))
    return [parse_one(b) for b in blocks if b.strip()]

def parse_one(block):
    given, when, then_ = [], [], []
    cur = None
    for line in block.splitlines():
        l = line.lower()
        if l.startswith("given"):   cur, given = "given",   [line[6:].strip()]
        elif l.startswith("and"):   given.append(line[4:].strip()) if cur=="given" else \
                                    when.append(line[4:].strip())  if cur=="when" else \
                                    then_.append(line[4:].strip())
        elif l.startswith("when"):  cur, when = "when",    [line[5:].strip()]
        elif l.startswith("then"):  cur, then_ = "then",   [line[5:].strip()]
    return {"given":given,"when":when,"then":then_,"raw":block}

# -------------------  CLASSIFIER  -------------------
def priority(sc):
    txt = sc["raw"].lower()
    if any(k in txt for k in MUST_KEYWORDS):      return "M"
    if any(k in txt for k in SHOULD_KEYWORDS):    return "S"
    if any(k in txt for k in COULD_ENHANCEMENTS): return "C"
    # most of the original stories are Must
    return "M"

# -------------------  GROUPER  -------------------
GROUP_RULES = [
    ("Display of Places",               ["places tab","display","following"]),
    ("Adding a Place to an Asset Item", ["insert a row","select","place","role","start date","end date","lock"]),
    ("Parent-Child Asset Place Propagation", ["child","apply the changes","yes","no","end date"]),
    ("Hierarchical Consistency Warning",["warning","parent","location"]),
]

def group_key(sc):
    txt = sc["raw"].lower()
    for name, words in GROUP_RULES:
        if all(w in txt for w in words): return name
    return "Other Requirements"

# -------------------  MEANING EXTRACTOR  -------------------
def meaning(sc):
    then = " ".join(sc["then"])
    when = " ".join(sc["when"])
    given = " ".join(sc["given"])

    # clean Navigate Cloud
    then = re.sub(r"Navigate Cloud\s*", "", then, flags=re.I)

    # special cases – keep the wording you liked
    if "display" in then.lower() and "place name" in then.lower():
        return "The Places tab **MUST** display: **Place Name, Role, Start Date, End Date**"
    if "insert a row" in then.lower():
        return "The system **MUST** allow inserting a new row to link a place"
    if "quickview" in then.lower():
        return "The system **MUST** provide a **quick view** of the selected place"
    if "search" in then.lower() and "select" in then.lower():
        return "Allow **search and select** existing place when adding"
    if "add a new place" in then.lower():
        return "Allow **creating a new place** as alternative to search"
    if "role" in then.lower() and "mandatory" in then.lower():
        return "Require **Role (Enum)** and **Start Date** as mandatory"
    if "end date" in then.lower() and "optional" in then.lower():
        return "Make **End Date optional** with validation: ≥ Start Date, not before today"
    if "lock" in then.lower() and "cannot be updated" in then.lower():
        return "After save, **lock Place, Role, Start Date**"
    if "edit the end date" in then.lower():
        return "Allow editing **End Date only** if empty or future"
    if "lock the whole row" in then.lower():
        return "If End Date ≤ today, **lock entire row**"
    if "apply the changes to child" in then.lower():
        return "Prompt: **Apply changes to child assets?** on add/end-date"
    if "select yes" in then.lower():
        return "On **Yes**: propagate place/role (add or update dates) to children"
    if "select no" in then.lower():
        return "On **No**: Save **only parent**, no child changes"
    if "warning" in then.lower() and "parent" in given.lower():
        return "Show warning if child place differs from active parent place"

    # fallback – clean sentence
    then = re.sub(r"allows me to |returns |displays |the following.*?:?", "", then)
    return then.strip(". ").capitalize()

# -------------------  BUILD HIERARCHICAL DOC  -------------------
def build_moscow_doc(scenarios):
    groups = OrderedDict()
    for s in scenarios:
        g = group_key(s)
        groups.setdefault(g, []).append(s)

    lines = ['# MoSCoW Requirements: Asset Item – Places Tab',
             '*(M) = Must, (S) = Should, (C) = Could, (W) = Won’t*','']

    req_no = 1
    for grp_name, items in groups.items():
        prio = priority(items[0])                     # all items in a group have same priority
        lines.append(f"### **({prio}) {grp_name}**")
        lines.append("")

        seen = set()
        for s in items:
            txt = meaning(s)
            if txt in seen: continue
            seen.add(txt)
            lines.append(f"{req_no}. **({prio})** {txt}")
            req_no += 1
        lines.append("---")

    # ---- Won’t Have (static) ----
    lines.append("### **(W) Out of Scope (Won’t Have)**")
    wont = [
        "Editing locked fields (Place, Role, Start Date) after initial save",
        "Overlap validation between multiple active place assignments",
        "Reassigning or replacing an existing active place (only add/end-date supported)"
    ]
    for i, w in enumerate(wont, req_no):
        lines.append(f"{i}. **(W)** {w}")
    lines.append("")

    lines.append("**Key:**  \n**(M)** = Critical for release  \n**(S)** = Important but not release-blocking  \n**(C)** = Desirable enhancement  \n**(W)** = Explicitly excluded")
    return "\n".join(lines)

# -------------------  DOWNLOAD HELPERS  -------------------
def download_link(data, filename, kind='excel'):
    b64 = base64.b64encode(data).decode()
    mime = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet' if kind=='excel' else 'text/csv'
    return f'<a href="data:{mime};base64,{b64}" download="{filename}">Download {filename}</a>'

# -------------------  UI  -------------------
uploader = widgets.FileUpload(accept='.xlsx,.csv', multiple=False,
                              description='Upload Excel/CSV')
out = widgets.Output()

def on_upload(change):
    with out:
        out.clear_output()
        if not uploader.value:
            print("No file selected.")
            return

        info = list(uploader.value.values())[0]
        name = info['metadata']['name']
        bytes_io = io.BytesIO(info['content'])

        # ----- read file -----
        if name.endswith('.csv'):
            df = pd.read_csv(bytes_io, encoding='utf-8', on_bad_lines='skip')
        else:
            df = pd.read_excel(bytes_io)

        if 'Acceptance Criteria' not in df.columns:
            print("Column **Acceptance Criteria** not found!")
            return

        # ----- process each row -----
        moscow_docs = []
        for crit in df['Acceptance Criteria']:
            if pd.isna(crit):
                moscow_docs.append("")
                continue
            scenarios = parse_gherkin(str(crit))
            doc = build_moscow_doc(scenarios)
            moscow_docs.append(doc)

        df['MoSCoW Requirements'] = moscow_docs

        # ----- show preview -----
        display(df.head())

        # ----- write back to memory -----
        buf = io.BytesIO()
        ext = 'xlsx' if name.endswith('.xlsx') else 'csv'
        if ext=='xlsx':
            df.to_excel(buf, index=False, engine='openpyxl')
        else:
            df.to_csv(buf, index=False)
        buf.seek(0)

        link = download_link(buf.getvalue(),
                             f"MoSCoW_{name.rsplit('.',1)[0]}.{ext}",
                             'excel' if ext=='xlsx' else 'csv')
        display(HTML("<br>" + link))

uploader.observe(on_upload, names='value')
display(uploader, out)